In [1]:
import sys
sys.path.insert(0, '/home/ethantu/good-vibrations')

from composer import Trainer
from composer.core import Evaluator
from composer.loggers import InMemoryLogger

from src2.model import VibrationTransformer
from src2.dataset import build_dataset
from src2.callbacks import OutputSaver

/home/ethantu/good-vibrations/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
train_loader, eval_loader, _ = build_dataset()
data_loaders = eval_loader+[Evaluator(label='train', dataloader=train_loader)]

Loaded dataset with 551 samples

x positions: [0 1 2 3 4 5 6 7 8 9 10 None]
y positions: [0 1 2 3 4 5 6 7 8 9 10 11 None]

Filtering dataset...
Final dataset contains 551 samples

Mask paths: ['image/cube-000x-001y-1obj-cardboard-2026-03-31-18-21-24/mask.npz', 'image/cube-000x-001y-1obj-cardboard-2026-03-31-18-21-22/mask.npz', 'image/cube-000x-001y-1obj-cardboard-2026-03-31-18-21-20/mask.npz', 'image/cube-000x-001y-1obj-cardboard-2026-03-31-18-21-18/mask.npz', 'image/cube-000x-002y-1obj-cardboard-2026-03-31-18-27-14/mask.npz']...
FFT paths: ['data/0000001/speckle_shifts_fft.npz', 'data/0000002/speckle_shifts_fft.npz', 'data/0000003/speckle_shifts_fft.npz', 'data/0000004/speckle_shifts_fft.npz', 'data/0000005/speckle_shifts_fft.npz']...


Fetching 1039 files:   0%|          | 0/1039 [00:00<?, ?it/s]

Downloaded snapshot to /home/ethantu/.cache/huggingface/hub/datasets--eturok-weizmann--laser-vibrations/snapshots/cd3612c5801878139c8e3edc5410e2088920c7c4

Loading masks and FFTs...
masks.shape=torch.Size([551, 956, 1069])	masks.dtype=torch.float32
fft.shape=torch.Size([551, 100, 3421, 2])	fft.dtype=torch.complex64

Discretizing masks...
masks.shape=torch.Size([551, 40, 20])	masks.dtype=torch.float32

Normalizing and patchifying FFTs...
fft.shape=torch.Size([551, 100, 13, 2, 256])	self.fft.dtype=torch.complex64

held_out_positions={(3, 4), (2, 2), (6, 6), (1, 9), (7, 8)}
551 total samples	416 train samples	105 eval samples	20 unseen position eval samples	10 multi-object eval samples

train_indices=[547, 189, 452, 353, 432, 294, 165, 442, 230, 376, 534, 501, 356, 293, 471, 380, 94, 204, 75, 530, 15, 234, 167, 196, 422, 19, 451, 56, 265, 418, 461, 170, 25, 274, 42, 336, 523, 366, 410, 134, 358, 31, 121, 438, 237, 425, 57, 528, 24, 17, 259, 66, 243, 390, 102, 221, 517, 23, 342, 287, 445, 

In [3]:
LOAD_PATH = '/home/ethantu/good-vibrations/runs/1778026819-greedy-caracara/checkpoints/ep100-ba700-rank0.pt'
trainer = Trainer(model=VibrationTransformer(), load_path=LOAD_PATH, loggers=InMemoryLogger(), callbacks=OutputSaver("1ep", "dummy"), progress_bar=False)

/home/ethantu/good-vibrations/.venv/lib/python3.12/site-packages/composer/trainer/trainer.py:1224: UserWarning: No optimizer was specified. Defaulting to DecoupledSGDW(lr=0.1)
  warnings.warn((


In [4]:
trainer.eval(data_loaders)
trainer.close()
del data_loaders
trainer.logger.destinations[0].data.keys()

/home/ethantu/good-vibrations/.venv/lib/python3.12/site-packages/composer/core/data_spec.py:40: UserWarning: Cannot split tensor of length 41 into batches of size 64. As it is smaller, no splitting will be done. This may happen on the last batch of a dataset if it is a smaller size than the microbatch size.
  warnings.warn(
/home/ethantu/good-vibrations/.venv/lib/python3.12/site-packages/composer/core/data_spec.py:29: UserWarning: Cannot split list of length 41 into batches of size 64. As it is smaller, no splitting will be done. This may happen on the last batch of a dataset if it is a smaller size than the microbatch size.
  warnings.warn(
/home/ethantu/good-vibrations/.venv/lib/python3.12/site-packages/composer/core/data_spec.py:40: UserWarning: Cannot split tensor of length 20 into batches of size 64. As it is smaller, no splitting will be done. This may happen on the last batch of a dataset if it is a smaller size than the microbatch size.
  warnings.warn(
/home/ethantu/good-vibra

dict_keys(['eval/base/fft', 'eval/base/mask_pred', 'eval/base/mask_true', 'eval/base/info', 'metrics/eval/base/mse', 'eval/unseen_pos/fft', 'eval/unseen_pos/mask_pred', 'eval/unseen_pos/mask_true', 'eval/unseen_pos/info', 'metrics/eval/unseen_pos/mse', 'eval/multi_object/fft', 'eval/multi_object/mask_pred', 'eval/multi_object/mask_true', 'eval/multi_object/info', 'metrics/eval/multi_object/mse', 'train/fft', 'train/mask_pred', 'train/mask_true', 'train/info', 'metrics/train/mse'])

In [5]:
def parse(label, outputs):
    out = {k:v for k, v in outputs.items() if label in k}
    return out | {f'{label}/{k}':v for k, v in out.pop(f'{label}/info').items()}

In [6]:
data = trainer.logger.destinations[0].most_recent_values
data = {k.replace('metrics/', ''): v for k, v in data.items()}
info_keys = [k for k in data.keys() if 'info' in k]
for info_key in info_keys: data |= {f'{info_key}/{k}':v for k, v in data.pop(info_key).items()}

print(data.keys())
for k, v in data.items(): print(f'{k}\t\t{v.shape if hasattr(v, 'shape') else v}')

dict_keys(['eval/base/fft', 'eval/base/mask_pred', 'eval/base/mask_true', 'eval/base/mse', 'eval/unseen_pos/fft', 'eval/unseen_pos/mask_pred', 'eval/unseen_pos/mask_true', 'eval/unseen_pos/mse', 'eval/multi_object/fft', 'eval/multi_object/mask_pred', 'eval/multi_object/mask_true', 'eval/multi_object/mse', 'train/fft', 'train/mask_pred', 'train/mask_true', 'train/mse', 'eval/base/info/sample_id', 'eval/base/info/x_position', 'eval/base/info/y_position', 'eval/base/info/n_objects', 'eval/base/info/speakers', 'eval/unseen_pos/info/sample_id', 'eval/unseen_pos/info/x_position', 'eval/unseen_pos/info/y_position', 'eval/unseen_pos/info/n_objects', 'eval/unseen_pos/info/speakers', 'eval/multi_object/info/sample_id', 'eval/multi_object/info/x_position', 'eval/multi_object/info/y_position', 'eval/multi_object/info/n_objects', 'eval/multi_object/info/speakers', 'train/info/sample_id', 'train/info/x_position', 'train/info/y_position', 'train/info/n_objects', 'train/info/speakers'])
eval/base/fft	

In [7]:
import pandas as pd
rows = []
tensor_store = []
for split in ["train", "eval/base", "eval/unseen_pos", "eval/multi_object"]:
  n = len(data[f"{split}/info/sample_id"])
  for i in range(n):
    tensor_store.append({
      "fft": data[f"{split}/fft"][i],
      "mask_pred": data[f"{split}/mask_pred"][i],
      "mask_true": data[f"{split}/mask_true"][i],
    })
    rows.append({
      "tensor_idx": len(tensor_store) - 1,
      "split": split,
      "mse": data[f"{split}/mse"][i] if data[f"{split}/mse"].ndim > 0 else data[f"{split}/mse"].item(),
      "sample_id": data[f"{split}/info/sample_id"][i].item(),
      "x_position": data[f"{split}/info/x_position"][i].item(),
      "y_position": data[f"{split}/info/y_position"][i].item(),
      "n_objects": data[f"{split}/info/n_objects"][i].item(),
      "speakers": data[f"{split}/info/speakers"][i],
    })
df = pd.DataFrame(rows)

In [8]:
df

,tensor_idx,split,mse,sample_id,x_position,y_position,n_objects,speakers
0,0,train,0.010910,234,5,6,1,0010
1,1,train,0.010910,480,10,7,1,1000
2,2,train,0.010910,243,5,8,1,0100
3,3,train,0.010910,463,10,3,1,0100
4,4,train,0.010910,282,6,7,1,0010
...,...,...,...,...,...,...,...,...
98,98,eval/multi_object,0.021091,540,-1,-1,5,1000
99,99,eval/multi_object,0.021091,541,-1,-1,5,0001
100,100,eval/multi_object,0.021091,542,-1,-1,5,0010
101,101,eval/multi_object,0.021091,543,-1,-1,5,0100
